# WavePID Analysis Example

This notebook demonstrates how to load and plot the ROOT output from the WavePID simulation.

## Prerequisites

Generate output first:
```bash
./OMSim_WavePID_study -n 10 --detector_type 3 --environment 2 -d 5 -e 30 -p mu- -o wavepid_example
```

Install Python dependencies:
```bash
pip install uproot awkward matplotlib numpy
```

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import os

# === SET YOUR ROOT FILE PATH HERE ===
filename = "wavepid_example_hits.root"

if not os.path.exists(filename):
    raise FileNotFoundError(
        f"ROOT file not found: {os.path.abspath(filename)}\n\n"
        "Generate it first by running:\n"
        "  ./OMSim_WavePID_study -n 10 --detector_type 3 --environment 2 "
        "-d 5 -e 30 -p mu- -o wavepid_example\n\n"
        "Then either copy the output file here or update the 'filename' variable above."
    )

file = uproot.open(filename)
tree = file["PhotonHits"]
data = tree.arrays(library="np")

In [ ]:
print(f"Total photon hits: {len(data['hitTime'])}")
print(f"Available branches: {list(data.keys())}")
print(f"Number of events: {len(np.unique(data['eventID']))}")

## Fig 3 — Stacked Tier Histogram

Stacked histogram of time residuals Δt, colored by photon origin tier.
The y-axis shows mean photons per event per ns.

**Time residual** Δt = t_hit − (t_c + t₀), where t_c is the direct Cherenkov photon travel time
and t₀ is the muon travel time to the emission point. Δt ≈ 0 for a direct Cherenkov photon.

**Tiers** are classified by `parentProcess` and `parentID`:
- **Primary particle** — Cerenkov from the primary muon (`parentProcess=muIoni`, `parentID=1`)
- **Primary-induced secondaries** — Cerenkov from secondary electrons/muons (`eIoni`, or `muIoni` with `parentID>1`)
- **Brems-secondaries** — compt, conv, ePairProd, muPairProd
- **Other**

> **Note:** Our simulation stores `parentProcess="muIoni"` for ALL Cerenkov from muons (primary and secondary alike),
> so `parentID==1` is used to identify the primary muon.

In [ ]:
# --- Geometry parameters: must match your simulation ---
d = 5.0               # impact parameter [m] — matches -d 5
C_VAC = 299.792458    # speed of light [mm/ns]
N_ICE = 1.32          # group refractive index in ice
THETA_C = np.radians(42.0)   # Cherenkov angle in ice
DOM_RADIUS = 0.1651   # DOM radius [m]

v_photon = C_VAC / N_ICE   # photon group velocity [mm/ns]

# Direct Cherenkov photon path and travel time
s_c = d / np.sin(THETA_C) - DOM_RADIUS    # [m]
t_c = (s_c * 1e3) / v_photon              # [ns]

# Muon path component (muon starts at x = -1.7*d in the simulation)
s_0 = d * (1.7 - 1.0 / np.tan(THETA_C))  # [m]
t_0 = (s_0 * 1e3) / C_VAC                 # [ns]

# Time residual for each hit
t_hit = data["hitTime"]
t_residual = t_hit - (t_c + t_0)

# --- Tier classification ---
# Note: our simulation hardcodes parentProcess="muIoni" for ALL Cerenkov from muons,
# including the primary. Use parentID==1 to identify the primary muon directly.
parent_process = np.array([str(pp) for pp in data["parentProcess"]])
parent_id = data["parentID"]

BREMS_PROCESSES = {"compt", "conv", "ePairProd", "muPairProd"}

mask_primary   = (parent_process == "muIoni") & (parent_id == 1)
mask_secondary = ((parent_process == "eIoni") |
                  ((parent_process == "muIoni") & (parent_id != 1)))
mask_brems     = np.array([pp in BREMS_PROCESSES for pp in parent_process])
mask_other     = ~(mask_primary | mask_secondary | mask_brems)

tier_masks = [
    ("Primary particle",             mask_primary,   "#0072B2"),
    ("Primary-induced secondaries",  mask_secondary, "#56B4E9"),
    ("Brems-secondaries",            mask_brems,     "#D55E00"),
    ("Other",                        mask_other,     "0.60"),
]

# --- Histogram ---
n_events = len(np.unique(data["eventID"]))
bin_width = 1.0
bins = np.arange(0.0, 17.0, bin_width)

data_list, color_list, label_list, weights_list = [], [], [], []
for label, mask, color in tier_masks:
    arr = t_residual[mask]
    arr = arr[np.isfinite(arr)]
    if len(arr) > 0:
        data_list.append(arr)
        color_list.append(color)
        label_list.append(label)
        weights_list.append(np.ones(len(arr)) / (n_events * bin_width))

fig, ax = plt.subplots(figsize=(6.69, 4.4))
ax.hist(data_list, bins=bins, stacked=True, color=color_list,
        edgecolor="none", alpha=0.85, weights=weights_list, zorder=2, label=label_list)

ax.set_xlabel("Residual time [ns]")
ax.set_ylabel("Photons [ns\u207b\u00b9]")
ax.set_title(r"$\mu^-$ (30 GeV), $d$ = 5 m")
ax.set_xticks(np.arange(0, 17, 2))
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.18, lw=0.8)
plt.tight_layout()
plt.show()

## Fig 4 — PDF Stairs + Average Cumulative Charge

Left axis (solid): normalized PDF of time residuals Δt as a step histogram.

Right axis (dashed): average cumulative photon count (PE) as a function of Δt,
i.e. the mean number of photons per event with residual time ≤ Δt.

In [ ]:
bins_pdf = np.arange(0.0, 17.0, 1.0)
bin_width = 1.0

# Filter to valid residuals in [0, 16) ns
event_ids = data["eventID"]
valid = np.isfinite(t_residual)
t_res_v = t_residual[valid]
eids_v = event_ids[valid]
unique_events = np.unique(eids_v)

# Normalized PDF
counts, _ = np.histogram(t_res_v, bins=bins_pdf)
pdf = counts / (counts.sum() * bin_width) if counts.sum() > 0 else counts.astype(float)

# Average cumulative charge: for each time edge, mean hits per event with t_res <= t
cum_avg = np.array([
    np.mean([np.sum(t_res_v[eids_v == eid] <= t) for eid in unique_events])
    for t in bins_pdf
])

# --- Plot ---
color = "tab:blue"
fig, ax1 = plt.subplots(figsize=(6.69 * 1.3, 3.6))
ax2 = ax1.twinx()

# Left axis: PDF
ax1.stairs(pdf, bins_pdf, color=color, lw=1.8, label=r"$\mu^-$ PDF", zorder=2)
ax1.fill_between(bins_pdf, np.append(pdf, pdf[-1]),
                 step="post", color=color, alpha=0.15, lw=0, zorder=1)

# Right axis: cumulative charge
ax2.plot(bins_pdf, cum_avg, color=color, ls="--", lw=1.8, alpha=0.8, zorder=3,
         drawstyle="steps-post", label=r"$\mu^-$ avg. cumulative charge")

ax1.set_xlabel(r"Residual time $\Delta t_{\gamma}$ [ns]")
ax1.set_ylabel(r"$p(\Delta t_{\gamma})$ [ns$^{-1}$]")
ax2.set_ylabel("Avg. cumulative charge [PE]", rotation=270, labelpad=22)
ax1.set_xticks(np.arange(0, 17, 4))
if cum_avg.max() > 0:
    ax2.set_ylim(0, cum_avg.max() * 1.05)
ax1.grid(axis="y", alpha=0.18, lw=0.8)
ax2.spines["top"].set_visible(False)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           bbox_to_anchor=(1.18, 0.5), loc="center left", fontsize=11)
fig.subplots_adjust(right=0.68, top=0.95)
plt.show()